# FingerNet: Inferência Bloco a Bloco

Este notebook executa a FingerNet passo a passo, exibindo **todos** os outputs intermediários:
- Mapas de ativação da VGG (backbone)
- Branches ASPP (orientação e segmentação)
- Banco de filtros de Gabor (kernels aprendidos)
- Módulo de Enhancement (filtragem, pico de orientação, fase)
- MinutiaeHead (features intermediárias e outputs)
- Pós-processamento final

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import fingernet
from fingernet.wrapper import postprocess
from fingernet.plot import plot_ori_field, plot_mnt


def show_activation_grid(tensor, title, num_cols=8, max_channels=32, cmap='viridis'):
    """Exibe um grid de mapas de ativação de um tensor (B, C, H, W)."""
    t = tensor.squeeze(0).detach().cpu().numpy()
    n = min(t.shape[0], max_channels)
    num_rows = (n + num_cols - 1) // num_cols
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(2 * num_cols, 2 * num_rows))
    if num_rows == 1:
        axes = axes[np.newaxis, :]
    for i in range(num_rows * num_cols):
        ax = axes[i // num_cols, i % num_cols]
        if i < n:
            ax.imshow(t[i], cmap=cmap)
            ax.set_title(f'ch {i}', fontsize=7)
        ax.axis('off')
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def show_single(tensor, title, cmap='gray'):
    """Exibe um tensor 2D ou (1, 1, H, W) como imagem."""
    t = tensor.squeeze().detach().cpu().numpy()
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.imshow(t, cmap=cmap)
    ax.set_title(title)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


def show_side_by_side(tensors, titles, cmap='gray'):
    """Exibe múltiplos tensores lado a lado."""
    n = len(tensors)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]
    for ax, t, title in zip(axes, tensors, titles):
        arr = t.squeeze().detach().cpu().numpy()
        ax.imshow(arr, cmap=cmap)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 1. Carregar Modelo e Imagem

In [ ]:
# === CONFIGURE AQUI ===
IMAGE_PATH = '../../datasets/NISTSD27/images/B101L9U.bmp'  # Altere para sua imagem
# ======================

fnet_wrapper = fingernet.get_fingernet()
model = fnet_wrapper.fingernet  # Modelo core (sem wrapper)
device = next(model.parameters()).device
print(f'Device: {device}')

# Carregar e preparar imagem
img_pil = Image.open(IMAGE_PATH).convert('L')
img_np = np.array(img_pil, dtype=np.float32) / 255.0
input_tensor = torch.from_numpy(img_np).unsqueeze(0).unsqueeze(0).to(device)  # (1, 1, H, W)

# Padding para múltiplo de 8 (igual ao wrapper)
_, _, h, w = input_tensor.shape
pad_h = (8 - h % 8) % 8
pad_w = (8 - w % 8) % 8
input_tensor = F.pad(input_tensor, (0, pad_w, 0, pad_h), mode='constant', value=0)

print(f'Shape original: ({h}, {w}) | Padded: {input_tensor.shape}')
show_single(input_tensor, f'Imagem de Entrada ({IMAGE_PATH})')

## 2. ImgNormalization

In [ ]:
with torch.no_grad():
    x_norm = model.img_norm(input_tensor)

print(f'Input  - min: {input_tensor.min():.4f}, max: {input_tensor.max():.4f}, mean: {input_tensor.mean():.4f}')
print(f'Normed - min: {x_norm.min():.4f}, max: {x_norm.max():.4f}, mean: {x_norm.mean():.4f}')

show_side_by_side(
    [input_tensor, x_norm],
    ['Entrada Original', 'Após ImgNormalization']
)

## 3. VGG Backbone (FeatureExtractor) - Block 1

- `conv1_1`: 1 → 64 canais, kernel 3x3
- `conv1_2`: 64 → 64 canais, kernel 3x3
- `pool1`: MaxPool 2x2 → resolução H/2, W/2

In [ ]:
fe = model.feature_extractor

with torch.no_grad():
    x1 = fe.conv1_1(x_norm)  # (1, 64, H, W)
    x2 = fe.conv1_2(x1)     # (1, 64, H, W)
    x3 = fe.pool1(x2)       # (1, 64, H/2, W/2)

print(f'conv1_1: {x1.shape} | conv1_2: {x2.shape} | pool1: {x3.shape}')
show_activation_grid(x1, 'VGG Block 1 - conv1_1 (64 canais)', num_cols=8, max_channels=64)
show_activation_grid(x2, 'VGG Block 1 - conv1_2 (64 canais)', num_cols=8, max_channels=64)
show_activation_grid(x3, 'VGG Block 1 - após pool1 (64 canais, H/2)', num_cols=8, max_channels=64)

## 4. VGG Backbone - Block 2

- `conv2_1`: 64 → 128 canais
- `conv2_2`: 128 → 128 canais
- `pool2`: MaxPool → resolução H/4, W/4

In [ ]:
with torch.no_grad():
    x4 = fe.conv2_1(x3)  # (1, 128, H/2, W/2)
    x5 = fe.conv2_2(x4)  # (1, 128, H/2, W/2)
    x6 = fe.pool2(x5)    # (1, 128, H/4, W/4)

print(f'conv2_1: {x4.shape} | conv2_2: {x5.shape} | pool2: {x6.shape}')
show_activation_grid(x4, 'VGG Block 2 - conv2_1 (128 canais)', num_cols=8, max_channels=32)
show_activation_grid(x6, 'VGG Block 2 - após pool2 (128 canais, H/4)', num_cols=8, max_channels=32)

## 5. VGG Backbone - Block 3

- `conv3_1`: 128 → 256 canais
- `conv3_2`: 256 → 256 canais
- `conv3_3`: 256 → 256 canais
- `pool3`: MaxPool → resolução H/8, W/8

In [ ]:
with torch.no_grad():
    x7 = fe.conv3_1(x6)   # (1, 256, H/4, W/4)
    x8 = fe.conv3_2(x7)   # (1, 256, H/4, W/4)
    x9 = fe.conv3_3(x8)   # (1, 256, H/4, W/4)
    features = fe.pool3(x9)  # (1, 256, H/8, W/8)

print(f'conv3_1: {x7.shape} | conv3_2: {x8.shape} | conv3_3: {x9.shape} | pool3: {features.shape}')
show_activation_grid(x7, 'VGG Block 3 - conv3_1 (256 canais)', num_cols=8, max_channels=32)
show_activation_grid(features, 'VGG Block 3 - features finais (256 canais, H/8)', num_cols=8, max_channels=32)

## 6. OrientationSegmentationHead (ASPP)

Três branches paralelas com diferentes taxas de dilatação (1, 4, 8).
Cada branch produz tanto orientação (90 canais) quanto segmentação (1 canal).
Os resultados são somados e passados por sigmoid.

In [ ]:
osh = model.ori_seg_head

with torch.no_grad():
    # Branch 1 (dilation=1)
    a1 = osh.atrous_1(features)
    o1 = osh.ori_branch_1(a1)   # (1, 90, H/8, W/8)
    s1 = osh.seg_branch_1(a1)   # (1, 1, H/8, W/8)

    # Branch 2 (dilation=4)
    a2 = osh.atrous_2(features)
    o2 = osh.ori_branch_2(a2)
    s2 = osh.seg_branch_2(a2)

    # Branch 3 (dilation=8)
    a3 = osh.atrous_3(features)
    o3 = osh.ori_branch_3(a3)
    s3 = osh.seg_branch_3(a3)

    # Combinação final
    ori_map = torch.sigmoid(o1 + o2 + o3)   # (1, 90, H/8, W/8)
    seg_map = torch.sigmoid(s1 + s2 + s3)   # (1, 1, H/8, W/8)

print(f'ori_map: {ori_map.shape} (min={ori_map.min():.3f}, max={ori_map.max():.3f})')
print(f'seg_map: {seg_map.shape} (min={seg_map.min():.3f}, max={seg_map.max():.3f})')

In [ ]:
# Visualizar as 3 branches de segmentação e o resultado combinado
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, data, title in zip(axes, 
    [torch.sigmoid(s1), torch.sigmoid(s2), torch.sigmoid(s3), seg_map],
    ['Seg Branch 1 (dil=1)', 'Seg Branch 2 (dil=4)', 'Seg Branch 3 (dil=8)', 'Seg Combinada (sigmoid)']):
    ax.imshow(data.squeeze().cpu().numpy(), cmap='hot', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')
plt.suptitle('Segmentação: Branches ASPP individuais vs Combinada', fontsize=14)
plt.tight_layout()
plt.show()

print(f'\nNota: seg_map é CONTÍNUA [0,1], não binária!')
print(f'Valores únicos arredondados a 1 casa: {torch.unique(torch.round(seg_map * 10) / 10).cpu().numpy()}')

In [ ]:
# Visualizar orientação: argmax de cada branch e do resultado combinado
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, data, title in zip(axes,
    [torch.sigmoid(o1), torch.sigmoid(o2), torch.sigmoid(o3), ori_map],
    ['Ori Branch 1 (dil=1)', 'Ori Branch 2 (dil=4)', 'Ori Branch 3 (dil=8)', 'Ori Combinada']):
    # argmax ao longo dos 90 canais → orientação dominante
    ori_idx = torch.argmax(data, dim=1).squeeze().cpu().numpy()
    ax.imshow(ori_idx, cmap='hsv')
    ax.set_title(title)
    ax.axis('off')
plt.suptitle('Orientação: argmax das Branches ASPP (0-89 = 0°-178°)', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Gabor Filter Bank (Kernels Aprendidos)

O EnhancementModule contém 90 filtros de Gabor reais e 90 imaginários (25x25),
um para cada orientação (0°-178° em passos de 2°).

In [ ]:
enh = model.enhancement_module

gabor_r = enh.gabor_real.weight.data.squeeze(1).cpu().numpy()  # (90, 25, 25)
gabor_i = enh.gabor_imag.weight.data.squeeze(1).cpu().numpy()  # (90, 25, 25)

print(f'Gabor Real: {gabor_r.shape} | Gabor Imag: {gabor_i.shape}')

# Grid 9x10 para os 90 kernels reais
fig, axes = plt.subplots(9, 10, figsize=(15, 14))
for i in range(90):
    ax = axes[i // 10, i % 10]
    ax.imshow(gabor_r[i], cmap='RdBu_r')
    ax.set_title(f'{i*2}°', fontsize=6)
    ax.axis('off')
fig.suptitle('Gabor Real Kernels (90 orientações, 25x25)', fontsize=14)
plt.tight_layout()
plt.show()

# Grid 9x10 para os 90 kernels imaginários
fig, axes = plt.subplots(9, 10, figsize=(15, 14))
for i in range(90):
    ax = axes[i // 10, i % 10]
    ax.imshow(gabor_i[i], cmap='RdBu_r')
    ax.set_title(f'{i*2}°', fontsize=6)
    ax.axis('off')
fig.suptitle('Gabor Imaginário Kernels (90 orientações, 25x25)', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Enhancement Module - Passo a Passo

1. Aplica filtros Gabor na imagem original (90 canais real + 90 imaginário)
2. Detecta pico de orientação dominante via convolução gaussiana circular
3. Seleciona a orientação máxima (one-hot)
4. Upsample 8x a orientação e modula os filtros Gabor
5. Calcula fase = atan2(imag, real)

In [ ]:
with torch.no_grad():
    # 1. Filtros Gabor aplicados à imagem ORIGINAL (não normalizada)
    filtered_real = enh.gabor_real(input_tensor)  # (1, 90, H, W)
    filtered_imag = enh.gabor_imag(input_tensor)  # (1, 90, H, W)

    # 2. Pico de orientação (convolução gaussiana circular)
    ori_peak_raw = enh._ori_highest_peak(ori_map)  # (1, 90, H/8, W/8)

    # 3. Selecionar orientação máxima (one-hot)
    ori_peak = enh._select_max_orientation(ori_peak_raw)  # (1, 90, H/8, W/8)

    # 4. Upsample da orientação para resolução original
    upsampled_ori = F.interpolate(ori_peak, scale_factor=8, mode='nearest')  # (1, 90, H, W)

    # 5. Modulação: soma ponderada dos filtros pela orientação
    enh_real = torch.sum(filtered_real * upsampled_ori, dim=1, keepdim=True)
    enh_imag = torch.sum(filtered_imag * upsampled_ori, dim=1, keepdim=True)

    # 6. Fase
    enhanced_phase = enh._atan2(enh_imag, enh_real)

print(f'filtered_real: {filtered_real.shape}')
print(f'filtered_imag: {filtered_imag.shape}')
print(f'ori_peak (one-hot): {ori_peak.shape}')
print(f'upsampled_ori: {upsampled_ori.shape}')
print(f'enh_real: {enh_real.shape}')
print(f'enhanced_phase: {enhanced_phase.shape}')

In [ ]:
# Amostras dos filtros Gabor aplicados (canais 0, 22, 45, 67)
sample_channels = [0, 22, 45, 67]
fig, axes = plt.subplots(2, len(sample_channels), figsize=(20, 10))
for j, ch in enumerate(sample_channels):
    axes[0, j].imshow(filtered_real[0, ch].cpu().numpy(), cmap='RdBu_r')
    axes[0, j].set_title(f'Gabor Real ch={ch} ({ch*2}°)')
    axes[0, j].axis('off')
    axes[1, j].imshow(filtered_imag[0, ch].cpu().numpy(), cmap='RdBu_r')
    axes[1, j].set_title(f'Gabor Imag ch={ch} ({ch*2}°)')
    axes[1, j].axis('off')
plt.suptitle('Filtros Gabor aplicados à imagem (amostras)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Orientação: pico, seleção e upsampled
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(torch.argmax(ori_map, dim=1).squeeze().cpu().numpy(), cmap='hsv')
axes[0].set_title('ori_map (argmax)')

axes[1].imshow(torch.argmax(ori_peak_raw, dim=1).squeeze().cpu().numpy(), cmap='hsv')
axes[1].set_title('Após Gaussian Peak Detection')

axes[2].imshow(torch.argmax(ori_peak, dim=1).squeeze().cpu().numpy(), cmap='hsv')
axes[2].set_title('Após Select Max (one-hot)')

axes[3].imshow(torch.argmax(upsampled_ori, dim=1).squeeze().cpu().numpy(), cmap='hsv')
axes[3].set_title('Upsampled 8x (resolução original)')

for ax in axes:
    ax.axis('off')
plt.suptitle('Pipeline de Orientação no Enhancement Module', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Resultados do Enhancement
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(enh_real.squeeze().cpu().numpy(), cmap='gray')
axes[0].set_title('Enhanced Real')

axes[1].imshow(enh_imag.squeeze().cpu().numpy(), cmap='gray')
axes[1].set_title('Enhanced Imaginary')

axes[2].imshow(enhanced_phase.squeeze().cpu().numpy(), cmap='twilight')
axes[2].set_title('Enhanced Phase (atan2)')

for ax in axes:
    ax.axis('off')
plt.suptitle('Enhancement Module - Outputs', fontsize=14)
plt.tight_layout()
plt.show()

## 9. MinutiaeHead - Passo a Passo

Entrada: `[enhanced_phase, upsampled_seg]` (2 canais) + `ori_map` (90 canais)

- 3 convoluções com pooling (2→64→128→256)
- 4 branches de saída: orientação (180), x_offset (8), y_offset (8), score (1)

In [ ]:
mh = model.minutiae_head

with torch.no_grad():
    # Preparar entrada (igual ao FingerNet.forward())
    upsampled_seg = F.interpolate(F.softsign(seg_map), scale_factor=8, mode='nearest')
    minutiae_input = torch.cat([enhanced_phase, upsampled_seg], dim=1)  # (1, 2, H, W)

    # Blocos convolucionais intermediários
    xm1 = mh.pool1(mh.conv1(minutiae_input))  # (1, 64, H/2, W/2)
    xm2 = mh.pool2(mh.conv2(xm1))             # (1, 128, H/4, W/4)
    mnt_features = mh.pool3(mh.conv3(xm2))     # (1, 256, H/8, W/8)

    # Branches de saída
    o_input = torch.cat([mnt_features, ori_map], dim=1)  # (1, 346, H/8, W/8)
    mnt_o = torch.sigmoid(mh.o_branch(o_input))    # (1, 180, H/8, W/8)
    mnt_w = torch.sigmoid(mh.w_branch(mnt_features))  # (1, 8, H/8, W/8)
    mnt_h = torch.sigmoid(mh.h_branch(mnt_features))  # (1, 8, H/8, W/8)
    mnt_s = torch.sigmoid(mh.s_branch(mnt_features))  # (1, 1, H/8, W/8)

print(f'Entrada minutiae: {minutiae_input.shape}')
print(f'conv1→pool1: {xm1.shape}')
print(f'conv2→pool2: {xm2.shape}')
print(f'conv3→pool3 (mnt_features): {mnt_features.shape}')
print(f'mnt_o: {mnt_o.shape} | mnt_w: {mnt_w.shape} | mnt_h: {mnt_h.shape} | mnt_s: {mnt_s.shape}')

In [ ]:
# Features intermediárias da MinutiaeHead
show_activation_grid(xm1, 'MinutiaeHead - conv1→pool1 (64 canais, H/2)', num_cols=8, max_channels=32)
show_activation_grid(xm2, 'MinutiaeHead - conv2→pool2 (128 canais, H/4)', num_cols=8, max_channels=32)
show_activation_grid(mnt_features, 'MinutiaeHead - conv3→pool3 (256 canais, H/8)', num_cols=8, max_channels=32)

In [ ]:
# Outputs da MinutiaeHead
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Score map
axes[0].imshow(mnt_s.squeeze().cpu().numpy(), cmap='hot', vmin=0, vmax=1)
axes[0].set_title('Score Map (mnt_s)')

# Orientação (argmax dos 180 canais)
axes[1].imshow(torch.argmax(mnt_o, dim=1).squeeze().cpu().numpy(), cmap='hsv')
axes[1].set_title('Minutiae Orientation (argmax 180ch)')

# X-offset (argmax dos 8 canais)
axes[2].imshow(torch.argmax(mnt_w, dim=1).squeeze().cpu().numpy(), cmap='viridis')
axes[2].set_title('X-offset (argmax 8ch)')

# Y-offset
axes[3].imshow(torch.argmax(mnt_h, dim=1).squeeze().cpu().numpy(), cmap='viridis')
axes[3].set_title('Y-offset (argmax 8ch)')

for ax in axes:
    ax.axis('off')
plt.suptitle('MinutiaeHead - Outputs', fontsize=14)
plt.tight_layout()
plt.show()

## 10. Pós-processamento e Resultados Finais

Usa a função `postprocess()` existente do wrapper para:
1. Binarizar e limpar a máscara de segmentação
2. Detectar minúcias (threshold + NMS)
3. Converter orientação para radianos e modular pela máscara
4. Normalizar imagem melhorada

In [ ]:
# Montar o dict de outputs no mesmo formato que FingerNet.forward()
upsampled_seg_out = F.interpolate(seg_map, scale_factor=8, mode='nearest')
raw_outputs = {
    'orientation upsample': upsampled_ori,
    'segmentation upsample': upsampled_seg_out,
    'segmentation': seg_map,
    'orientation': ori_map,
    'enhanced_real': enh_real,
    'enhanced_phase': enhanced_phase,
    'minutiae_orientation': mnt_o,
    'minutiae_x_offset': mnt_w,
    'minutiae_y_offset': mnt_h,
    'minutiae_score': mnt_s
}

# Pós-processamento completo (incluindo quality_mask e unmodulated)
final = postprocess(raw_outputs, threshold=0.5, quality_mask=True, unmodulated=True)

print('Chaves do resultado:', list(final.keys()))

In [ ]:
# Visualização final: outputs padrão
img_np_orig = input_tensor.squeeze().cpu().numpy()
seg_mask_np = final['segmentation_mask'].squeeze().cpu().numpy()
enh_np = final['enhanced_image'].squeeze().cpu().numpy()
ori_np = final['orientation_field'].squeeze().cpu().numpy()
mnt_np = final['minutiae'][0].cpu().numpy()

fig, axes = plt.subplots(1, 5, figsize=(25, 5))

axes[0].imshow(img_np_orig, cmap='gray')
axes[0].set_title('Entrada')

axes[1].imshow(seg_mask_np, cmap='gray')
axes[1].set_title('Máscara Binária')

axes[2].imshow(enh_np, cmap='gray')
axes[2].set_title('Enhanced (modulado)')

axes[3].imshow(img_np_orig, cmap='gray')
plot_ori_field(axes[3], ori_np, stride=12)
axes[3].set_title('Orientação (modulado)')

axes[4].imshow(img_np_orig, cmap='gray')
if len(mnt_np) > 0:
    plot_mnt(axes[4], mnt_np, r=12)
axes[4].set_title(f'Minúcias ({len(mnt_np)})')

for ax in axes:
    ax.axis('off')
plt.suptitle('Resultados Finais (padrão - modulados pela segmentação)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Quality mask vs máscara binária
qmask_np = final['quality_mask'].squeeze().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(seg_map.squeeze().cpu().numpy(), cmap='hot', vmin=0, vmax=1)
axes[0].set_title('Segmentação Raw (H/8, contínua [0,1])')

axes[1].imshow(qmask_np, cmap='hot', vmin=0, vmax=255)
axes[1].set_title('Quality Mask (upsampled bilinear, 0-255)')

axes[2].imshow(seg_mask_np, cmap='gray')
axes[2].set_title('Máscara Binária (round + blur + round)')

for ax in axes:
    ax.axis('off')
plt.suptitle('Segmentação: Contínua vs Binária', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Modulado vs Não-modulado
enh_unmod_np = final['enhanced_image_unmod'].squeeze().cpu().numpy()
ori_unmod_np = final['orientation_field_unmod'].squeeze().cpu().numpy()
mnt_unmod_np = final['minutiae_unmod'][0].cpu().numpy()

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Linha 1: Modulado
axes[0, 0].imshow(enh_np, cmap='gray')
axes[0, 0].set_title('Enhanced (modulado)')

axes[0, 1].imshow(img_np_orig, cmap='gray')
plot_ori_field(axes[0, 1], ori_np, stride=12)
axes[0, 1].set_title('Orientação (modulado)')

axes[0, 2].imshow(img_np_orig, cmap='gray')
if len(mnt_np) > 0:
    plot_mnt(axes[0, 2], mnt_np, r=12)
axes[0, 2].set_title(f'Minúcias moduladas ({len(mnt_np)})')

# Linha 2: Não-modulado
axes[1, 0].imshow(enh_unmod_np, cmap='gray')
axes[1, 0].set_title('Enhanced (não-modulado)')

axes[1, 1].imshow(img_np_orig, cmap='gray')
plot_ori_field(axes[1, 1], ori_unmod_np, stride=12)
axes[1, 1].set_title('Orientação (não-modulado)')

axes[1, 2].imshow(img_np_orig, cmap='gray')
if len(mnt_unmod_np) > 0:
    plot_mnt(axes[1, 2], mnt_unmod_np, r=12)
axes[1, 2].set_title(f'Minúcias não-moduladas ({len(mnt_unmod_np)})')

for ax in axes.flat:
    ax.axis('off')
plt.suptitle('Comparação: Modulado (com máscara) vs Não-Modulado', fontsize=14)
plt.tight_layout()
plt.show()